In [2]:
pip install pandas

  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.7 MB 10.4 MB/s eta 0:00:01
   ----------- ---------------------------- 2.9/9.7 MB 7.7 MB/s eta 0:00:01
   ----------------- ---------------------- 4.2/9.7 MB 7.2 MB/s eta 0:00:01
   ----------------------- ---------------- 5.8/9.7 MB 6.9 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.7 MB 6.8 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.7 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.7 MB 6.6 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 6.4 MB/s  0:00:01
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.3 MB 8.3 MB/s eta 0:00:02
   --------- ------------------------------ 2.9/12.3 MB 7.1 MB/s eta 0:00:02
   ------------- ----------------

In [1]:
import pandas as pd

campaigns = pd.read_csv("../data/raw/dim_campaigns.csv")
products  = pd.read_csv("../data/raw/dim_products.csv")
stores    = pd.read_csv("../data/raw/dim_stores.csv")
events    = pd.read_csv("../data/raw/fact_events.csv")

All csv file imported 

In [2]:
campaigns.info()
products.info()
stores.info()
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   campaign_id    2 non-null      str  
 1   campaign_name  2 non-null      str  
 2   start_date     2 non-null      str  
 3   end_date       2 non-null      str  
dtypes: str(4)
memory usage: 196.0 bytes
<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_code  15 non-null     str  
 1   product_name  15 non-null     str  
 2   category      15 non-null     str  
dtypes: str(3)
memory usage: 492.0 bytes
<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   store_id  50 non-null     str  
 1   city      50 non-null     str  
dtypes: str(2)
memory usage: 932.0 bytes
<

In [3]:
events.describe()
events.isnull().sum()

event_id                       0
store_id                       0
campaign_id                    0
product_code                   0
base_price                     0
promo_type                     0
quantity_sold(before_promo)    0
quantity_sold(after_promo)     0
dtype: int64

In [4]:
data = events.merge(campaigns, on="campaign_id") \
             .merge(products, on="product_code") \
             .merge(stores, on="store_id")

data.head()

,event_id,store_id,campaign_id,product_code,base_price,promo_type,quantity_sold(before_promo),quantity_sold(after_promo),campaign_name,start_date,end_date,product_name,category,city
0,7f650b,STCBE-2,CAMP_SAN_01,P11,190,50% OFF,34,52,Sankranti,10/1/2024,16-01-2024,Atliq_Doodh_Kesar_Body_Lotion (200ML),Personal Care,Coimbatore
1,a21f91,STBLR-8,CAMP_DIW_01,P03,156,25% OFF,393,322,Diwali,12/11/2023,18-11-2023,Atliq_Suflower_Oil (1L),Grocery & Staples,Bengaluru
2,78bc80,STVJD-0,CAMP_SAN_01,P07,300,BOGOF,22,85,Sankranti,10/1/2024,16-01-2024,Atliq_Curtains,Home Care,Vijayawada
3,a1503f,STCBE-1,CAMP_DIW_01,P15,3000,500 Cashback,329,1000,Diwali,12/11/2023,18-11-2023,Atliq_Home_Essential_8_Product_Combo,Combo1,Coimbatore
4,1091cf,STBLR-6,CAMP_DIW_01,P05,55,25% OFF,108,93,Diwali,12/11/2023,18-11-2023,Atliq_Scrub_Sponge_For_Dishwash,Home Care,Bengaluru


In [5]:
data.shape

(1500, 14)

everything is correct ✅

In [6]:
before_total = data["quantity_sold(before_promo)"].sum()
after_total  = data["quantity_sold(after_promo)"].sum()

growth = (after_total - before_total) / before_total * 100

before_total, after_total, growth

(np.int64(209050), np.int64(435473), np.float64(108.31045204496532))

108.31% growth noted after promotion

In [12]:
data["revenue_before"] = (
    data["base_price"] *
    data["quantity_sold(before_promo)"]
)

In [13]:
data["revenue_after"] = (
    data["base_price"] *
    data["quantity_sold(after_promo)"]
)

In [14]:
rev_before = data["revenue_before"].sum()
rev_after  = data["revenue_after"].sum()

rev_growth = (rev_after - rev_before) / rev_before * 100

rev_before, rev_after, rev_growth

(np.int64(140701188), np.int64(347860150), np.float64(147.23327140635087))

Promotions led to a ~147% increase in estimated revenue, significantly higher than unit growth (~108%), indicating that promotional campaigns not only increased sales volume but also drove higher-value purchases. However, revenue is calculated using base prices and does not account for discounts, so actual net revenue impact may differ.

In [16]:
combined_analysis = data.groupby("promo_type")[[
    "quantity_sold(after_promo)",
    "revenue_after"
]].sum().sort_values("revenue_after", ascending=False)

combined_analysis

,quantity_sold(after_promo),revenue_after
promo_type,,
500 Cashback,63180,189540000
BOGOF,215253,95244220
33% OFF,90576,52204752
25% OFF,38290,7998603
50% OFF,28174,2872575


Promotional effectiveness varies by business objective. 
Cashback offers generate the highest revenue despite lower sales volume, 
while BOGOF promotions drive the largest unit sales but comparatively lower revenue. 
Deep discounts (25%–50% OFF) deliver poor financial returns.

In [7]:
promo_analysis = data.groupby("promo_type")[[
    "quantity_sold(before_promo)",
    "quantity_sold(after_promo)"
]].sum()

promo_analysis["growth_%"] = (
    (promo_analysis["quantity_sold(after_promo)"]
     - promo_analysis["quantity_sold(before_promo)"])
    / promo_analysis["quantity_sold(before_promo)"] * 100
)

promo_analysis.sort_values("growth_%", ascending=False)

,quantity_sold(before_promo),quantity_sold(after_promo),growth_%
promo_type,,,
BOGOF,58180,215253,269.977656
500 Cashback,22299,63180,183.331091
33% OFF,63321,90576,43.042593
50% OFF,21243,28174,32.627218
25% OFF,44007,38290,-12.991115


1) BOGOF is the BEST performer
2) Cashback also performs very well
3) High discounts ≠ best performance
4) 25% OFF is actually BAD

means - ✔ Best: BOGOF, Cashback
        ✔ Average: 33%, 50% OFF
        ❌ Worst: 25% OFF

Now lets see which camp perform better -

In [8]:
campaign_analysis = data.groupby("campaign_name")[[
    "quantity_sold(before_promo)",
    "quantity_sold(after_promo)"
]].sum()

campaign_analysis["growth_%"] = (
    (campaign_analysis["quantity_sold(after_promo)"]
     - campaign_analysis["quantity_sold(before_promo)"])
    / campaign_analysis["quantity_sold(before_promo)"] * 100
)

campaign_analysis

,quantity_sold(before_promo),quantity_sold(after_promo),growth_%
campaign_name,,,
Diwali,110319,183404,66.248788
Sankranti,98731,252069,155.308870


Sankranti campain is clearly winning 🥇 but why ? lets find out with more details.

In [9]:
top_products = data.groupby("product_name")[
    "quantity_sold(after_promo)"
].sum().sort_values(ascending=False)

top_products.head(10)

product_name
Atliq_Farm_Chakki_Atta (1KG)            81290
Atliq_Suflower_Oil (1L)                 74478
Atliq_Home_Essential_8_Product_Combo    63180
Atliq_Sonamasuri_Rice (10KG)            53235
Atliq_Masoor_Dal (1KG)                  37341
Atliq_High_Glo_15W_LED_Bulb             29928
Atliq_waterproof_Immersion_Rod          23685
Atliq_Curtains                          16317
Atliq_Double_Bedsheet_set               15058
Atliq_Lime_Cool_Bathing_Bar (125GM)     10280
Name: quantity_sold(after_promo), dtype: int64


INSIGHT 1: Most top products are: Grocery & Staples - Atta, Rice, Oil, Dal

INSIGHT 2: The Home Essential Combo leads at 136% IR — the 500 Cashback

INSIGHT 3: Top performers are NOT: Premium items Low-frequency products

In [10]:
category_analysis = data.groupby("category")[
    "quantity_sold(after_promo)"
].sum().sort_values(ascending=False)

category_analysis

category
Grocery & Staples    246344
Combo1                63180
Home Appliances       53613
Home Care             40832
Personal Care         31504
Name: quantity_sold(after_promo), dtype: int64

Key Business Insights:
🟢 1) Grocery & Staples Dominates by a Huge Margin
         Nearly 4× higher than next category.

🟢 2) Combo Packs Are Strong
        Supports earlier observation:

🟡 3) Durable Goods Perform Moderately
        Home appliances and home care:
        Bought less frequently
        Less impulse-driven           

🔴 4) Personal Care Shows Weak Response
        Likely: Lower urgency, Brand loyalty, Smaller basket impact            

In [11]:
city_analysis = data.groupby("city")[
    "quantity_sold(after_promo)"
].sum().sort_values(ascending=False)

city_analysis

city
Bengaluru        105141
Chennai           83273
Hyderabad         69399
Coimbatore        38900
Mysuru            37470
Visakhapatnam     33916
Madurai           31169
Mangalore         14929
Vijayawada        11106
Trivandrum        10170
Name: quantity_sold(after_promo), dtype: int64

Key Business Insights
🟢 1) Bengaluru is the Top Market

🟢 2) Tier-1 Cities Drive Most Sales Top 3: Bengaluru, Chennai, Hyderabad

🟡 3) Mid-tier Cities Show Moderate Performance: Coimbatore, Mysuru, Vizag

🔴 4) Long Tail of Low-Performing Cities: Mangalore, Vijayawada, Trivandrum


final insights : 

1. Overall promotions increased sales by ~108%
2. BOGOF and Cashback are most effective promotions
3. 25% discount performed poorly (negative growth)
4. Sankranti campaign outperformed Diwali significantly
5. Grocery & Staples category dominates sales
6. Combo products perform strongly
7. Top cities: Bengaluru, Chennai, Hyderabad
8. Promotions work best for essential, high-frequency products